# Módulo 10 · Aula 03 — Alta Performance e Polars

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O script do relatório anual começou a dar `MemoryError`. Comprei mais memória para o servidor. Funcionou por dois meses. Agora dá de novo."*

Comprar memória é a resposta certa **às vezes**. Antes disso, vale entender por que 200 MB de CSV viram 2 GB na memória.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | Onde a memória vai | 🎯 Medido, coluna a coluna |
| 2 | **`dtypes`** | Reduções de 80% com uma linha |
| 3 | `category` | O ganho maior de todos |
| 4 | Leitura em pedaços | Quando não cabe |
| 5 | **Parquet** | O formato que resolve |
| 6 | PyArrow | O que está por baixo |
| 7 | **Polars preguiçoso** | 🎯 O plano de consulta |
| 8 | DuckDB | SQL sobre arquivo |
| 9 | O que escolher | E quando o Pandas basta |

> 🎯 **Tudo aqui é medido.** Nenhuma afirmação de desempenho sem número ao lado.

## ⚙️ Preparação

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 10
# ═══════════════════════════════════════════════════════════════
import json
import os
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import time
import warnings
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("pandas", "pandas"), ("numpy", "numpy"),
               ("pyarrow", "pyarrow"), ("polars", "polars"),
               ("duckdb", "duckdb")]:
    _garantir(_p, _m)

import numpy as np
import pandas as pd

TEM_POLARS = _garantir("polars", "polars")
TEM_DUCKDB = _garantir("duckdb", "duckdb")
TEM_ARROW = _garantir("pyarrow", "pyarrow")

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print(f"pandas {pd.__version__} · numpy {np.__version__}")
if TEM_POLARS:
    import polars as pl
    print(f"polars {pl.__version__}")
if TEM_DUCKDB:
    import duckdb
    print(f"duckdb {duckdb.__version__}")


# ═══════════════════════════════════════════════════════════════
#  Dados da Aurora — gerados de forma REPRODUTÍVEL
# ═══════════════════════════════════════════════════════════════
SEMENTE = 20260813
rng = np.random.default_rng(SEMENTE)
random.seed(SEMENTE)

CIDADES = ["Campinas", "São Paulo", "Valinhos", "Sumaré",
           "Indaiatuba", "Jundiaí", "Hortolândia"]
CANAIS = ["site", "app", "marketplace"]
CATEGORIAS = {
    "NB": ("Notebooks", 1800, 4200),
    "MO": ("Monitores", 700, 2200),
    "PE": ("Periféricos", 40, 400),
    "AR": ("Armazenamento", 180, 900),
}


def gerar_produtos(n: int = 60) -> pd.DataFrame:
    linhas = []
    for i in range(n):
        prefixo = list(CATEGORIAS)[i % len(CATEGORIAS)]
        categoria, minimo, maximo = CATEGORIAS[prefixo]
        preco = round(float(rng.uniform(minimo, maximo)), 2)
        linhas.append({
            "sku": f"{prefixo}-{1000 + i}",
            "nome": f"{categoria[:-1]} modelo {i:03d}",
            "categoria": categoria,
            "preco": preco,
            "custo": round(preco * float(rng.uniform(0.55, 0.85)), 2),
            "estoque": int(rng.integers(0, 200)),
        })
    return pd.DataFrame(linhas)


def gerar_vendas(n: int = 50_000, dias: int = 180,
                 produtos: pd.DataFrame | None = None) -> pd.DataFrame:
    """Vendas sintéticas com sazonalidade e um pouco de sujeira."""
    produtos = gerar_produtos() if produtos is None else produtos
    fim = datetime(2026, 8, 1, tzinfo=timezone.utc)
    inicio = fim - timedelta(days=dias)

    idx = rng.integers(0, len(produtos), n)
    escolhidos = produtos.iloc[idx].reset_index(drop=True)

    # 📈 Sazonalidade: mais vendas no fim de semana e no fim do mês
    deslocamento = rng.integers(0, dias, n)
    datas = pd.to_datetime(inicio) + pd.to_timedelta(deslocamento, unit="D")
    datas = datas + pd.to_timedelta(rng.integers(0, 86400, n), unit="s")

    return pd.DataFrame({
        "pedido_id": 100_000 + np.arange(n),
        "data": datas,
        "sku": escolhidos["sku"],
        "categoria": escolhidos["categoria"],
        "cidade": rng.choice(CIDADES, n, p=[.28, .22, .12, .12, .11, .09, .06]),
        "canal": rng.choice(CANAIS, n, p=[.55, .30, .15]),
        "quantidade": rng.integers(1, 6, n),
        "preco_unitario": escolhidos["preco"],
        "custo_unitario": escolhidos["custo"],
        "status": rng.choice(["pago", "pendente", "cancelado"], n, p=[.82, .10, .08]),
        "frete": np.round(rng.uniform(0, 45, n), 2),
    })


# ═══════════════════════════════════════════════════════════════
#  Medição
# ═══════════════════════════════════════════════════════════════

def cronometrar(funcao, repeticoes: int = 1):
    """Devolve (resultado, milissegundos_medios)."""
    inicio = time.perf_counter()
    resultado = None
    for _ in range(repeticoes):
        resultado = funcao()
    return resultado, (time.perf_counter() - inicio) * 1000 / repeticoes


def comparar(casos: list[tuple[str, callable]], repeticoes: int = 1,
             rotulo: str = "abordagem"):
    """Mede várias abordagens e mostra o ganho relativo."""
    medidos = []
    for nome, funcao in casos:
        _, ms = cronometrar(funcao, repeticoes)
        medidos.append((nome, ms))
    melhor = min(m for _, m in medidos)
    largura = max(len(n) for n, _ in medidos) + 2
    print(f"{rotulo:<{largura}}{'tempo':>12}   {'vs melhor':>10}")
    print("─" * (largura + 26))
    for nome, ms in medidos:
        barra = "█" * max(1, int(ms / melhor))
        print(f"{nome:<{largura}}{ms:>9.1f} ms   {ms / melhor:>8.1f}×  {barra[:26]}")
    return medidos


def tamanho(n: int) -> str:
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:,.1f} {unidade}" if unidade != "B" else f"{n:,} B"
        n /= 1024
    return ""


def memoria(df: pd.DataFrame) -> int:
    return int(df.memory_usage(deep=True).sum())


def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]) -> None:
    """Tabela ASCII alinhada (marcadores ASCII, não emoji — M03)."""
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


print("\n✅ `gerar_vendas()`, `comparar()`, `memoria()`, `tabela()` prontos")
print(f"   semente fixa ({SEMENTE}) — os números são reprodutíveis")

## 1. Onde a memória vai

In [ ]:
BASE = preparar("aula_10_03")
vendas = gerar_vendas(n=300_000, dias=365)

print(f"{len(vendas):,} vendas · {tamanho(memoria(vendas))} na memória\n")

uso = vendas.memory_usage(deep=True).drop("Index").sort_values(ascending=False)
total = uso.sum()
for coluna, bytes_ in uso.items():
    barra = "█" * max(1, int(bytes_ / total * 40))
    print(f"   {coluna:<18}{tamanho(bytes_):>12}  {bytes_/total*100:>5.1f}%  {barra}")

print(f"\n   {'TOTAL':<18}{tamanho(total):>12}")

> 🎯 **As colunas de texto dominam — e elas são as mais repetitivas.**
>
> `categoria` tem 4 valores distintos repetidos 300 mil vezes. `cidade` tem 7. Cada linha guarda um **ponteiro para um objeto `str` do Python**, e cada `str` carrega cerca de 50 bytes de cabeçalho.
>
> 💭 O Pandas 1.x/2.x guarda texto como `object`: um array de ponteiros para objetos espalhados na memória. Não é só grande — é lento, porque cada acesso é um salto de ponteiro.

In [ ]:
# Quanto custa UM texto em Python
exemplo = "Campinas"
print(f"   a palavra '{exemplo}' tem {len(exemplo)} caracteres")
print(f"   como str do Python ocupa {sys.getsizeof(exemplo)} bytes")
print(f"   + 8 bytes do ponteiro no array\n")
print(f"   × {len(vendas):,} linhas = "
      f"{tamanho((sys.getsizeof(exemplo) + 8) * len(vendas))}")
print("\n   💭 E são só 7 cidades diferentes. A informação real cabe em 3 bits.")

## 2. 🎯 `dtypes` — reduções com uma linha

In [ ]:
print("Os tipos numéricos, e a faixa de cada um:\n")
tipos = [
    ["int8",    "-128 a 127",                    "1 byte"],
    ["int16",   "-32.768 a 32.767",              "2 bytes"],
    ["int32",   "±2,1 bilhões",                  "4 bytes"],
    ["int64",   "±9,2 quintilhões",              "8 bytes ← o padrão"],
    ["float32", "~7 dígitos significativos",     "4 bytes"],
    ["float64", "~15 dígitos significativos",    "8 bytes ← o padrão"],
]
tabela(["TIPO", "FAIXA", "TAMANHO"], tipos, [10, 34, 24])

print("""
💭 O Pandas escolhe int64/float64 por segurança — ele não sabe os
   seus dados. `quantidade` vai de 1 a 5 e ocupa 8 bytes por linha.
""")

In [ ]:
def otimizar(df: pd.DataFrame, limite_categoria: float = 0.5) -> pd.DataFrame:
    """Reduz os tipos sem perder informação.

    🔴 A EXCEÇÃO IMPORTANTE: dinheiro NÃO vira float32.

       float32 tem ~7 dígitos significativos. Um valor como
       1_234_567.89 já não cabe — e o centavo se perde silenciosamente.
       Você viu isso no M05, ao escolher NUMERIC em vez de FLOAT.
    """
    saida = df.copy()
    COLUNAS_DE_DINHEIRO = {"preco_unitario", "custo_unitario", "frete",
                           "receita", "margem", "preco", "custo"}

    for coluna in saida.columns:
        tipo = saida[coluna].dtype

        if pd.api.types.is_integer_dtype(tipo):
            saida[coluna] = pd.to_numeric(saida[coluna], downcast="integer")

        elif pd.api.types.is_float_dtype(tipo):
            if coluna in COLUNAS_DE_DINHEIRO:
                continue                      # 🔴 dinheiro fica em float64
            saida[coluna] = pd.to_numeric(saida[coluna], downcast="float")

        elif tipo == object:
            distintos = saida[coluna].nunique()
            if distintos / len(saida) < limite_categoria:
                saida[coluna] = saida[coluna].astype("category")

    return saida


otimizado = otimizar(vendas)

antes, depois = memoria(vendas), memoria(otimizado)
print(f"   antes  : {tamanho(antes)}")
print(f"   depois : {tamanho(depois)}")
print(f"   redução: {(1 - depois / antes) * 100:.1f}%  "
      f"({antes / depois:.1f}× menor)\n")

comparacao = pd.DataFrame({
    "antes": vendas.dtypes.astype(str),
    "depois": otimizado.dtypes.astype(str),
    "bytes_antes": vendas.memory_usage(deep=True).drop("Index"),
    "bytes_depois": otimizado.memory_usage(deep=True).drop("Index"),
})
comparacao["reducao"] = (
    1 - comparacao["bytes_depois"] / comparacao["bytes_antes"]).map("{:.0%}".format)
print(comparacao[["antes", "depois", "reducao"]].to_string())

> 🔴 **Repare que `preco_unitario` e `custo_unitario` continuam `float64` — de propósito.**
>
> `float32` guarda cerca de 7 dígitos significativos. Um faturamento de `1.234.567,89` já não cabe: o valor é arredondado e o centavo desaparece **sem erro nenhum**.
>
> 💭 É a mesma decisão do M05 (`NUMERIC` em vez de `FLOAT` no Postgres) e do M01 (`0.1 + 0.2 != 0.3`). **Economizar memória em coluna de dinheiro é economizar no lugar errado.**

In [ ]:
# Prova rápida de que float32 perde centavo
valor = 1_234_567.89
print(f"   float64: {np.float64(valor):,.2f}")
print(f"   float32: {np.float32(valor):,.2f}   🔴 perdeu o centavo")

serie64 = pd.Series([valor] * 1000)
serie32 = serie64.astype("float32")
print(f"\n   soma de 1.000 valores:")
print(f"     float64: {serie64.sum():>18,.2f}")
print(f"     float32: {serie32.sum():>18,.2f}")
print(f"     🔴 erro: {abs(serie64.sum() - serie32.sum()):>18,.2f}")

In [ ]:
# 🔑 Melhor ainda: declarar os tipos NA LEITURA
CSV = BASE / "vendas.csv"
vendas.to_csv(CSV, index=False)

TIPOS = {
    "pedido_id": "int32",
    "sku": "category",
    "categoria": "category",
    "cidade": "category",
    "canal": "category",
    "status": "category",
    "quantidade": "int8",
    "preco_unitario": "float64",   # 🔴 dinheiro
    "custo_unitario": "float64",   # 🔴 dinheiro
    "frete": "float64",            # 🔴 dinheiro
}

def ler_ingenuo():
    return pd.read_csv(CSV, parse_dates=["data"])

def ler_com_tipos():
    return pd.read_csv(CSV, dtype=TIPOS, parse_dates=["data"])

print("Leitura ingênua vs com tipos declarados:\n")
comparar([("read_csv() simples", ler_ingenuo),
          ("read_csv(dtype=...)", ler_com_tipos)], rotulo="leitura")

print(f"\n   memória ingênua    : {tamanho(memoria(ler_ingenuo()))}")
print(f"   memória com tipos  : {tamanho(memoria(ler_com_tipos()))}")
print("""
🔑 Declarar os tipos na leitura evita o PICO de memória: sem eles, o
   Pandas carrega tudo como object/int64 primeiro e só depois você
   converte — e por um instante as duas versões coexistem.
""")

## 3. Leitura em pedaços — quando não cabe

In [ ]:
def em_pedacos(caminho: Path, tamanho_pedaco: int = 50_000) -> pd.Series:
    """Agrega sem carregar o arquivo inteiro.

    🔑 O padrão: processe pedaço, guarde só o AGREGADO, descarte o resto.
       A memória fica constante, independente do tamanho do arquivo.
    """
    acumulado = {}
    pedacos = 0
    for pedaco in pd.read_csv(caminho, dtype=TIPOS, parse_dates=["data"],
                              chunksize=tamanho_pedaco):
        pedacos += 1
        pedaco = pedaco[pedaco["status"] == "pago"]
        receita = pedaco["quantidade"] * pedaco["preco_unitario"]
        parcial = receita.groupby(pedaco["categoria"], observed=True).sum()
        for chave, valor in parcial.items():
            acumulado[chave] = acumulado.get(chave, 0.0) + valor
    print(f"   processado em {pedacos} pedaços de {tamanho_pedaco:,} linhas")
    return pd.Series(acumulado).sort_values(ascending=False)


def de_uma_vez(caminho: Path) -> pd.Series:
    df = pd.read_csv(caminho, dtype=TIPOS, parse_dates=["data"])
    df = df[df["status"] == "pago"]
    receita = df["quantidade"] * df["preco_unitario"]
    return receita.groupby(df["categoria"], observed=True).sum().sort_values(ascending=False)


resultado_pedacos = em_pedacos(CSV)
resultado_inteiro = de_uma_vez(CSV)

print(f"\n{'categoria':<18}{'em pedaços':>16}{'de uma vez':>16}")
print("─" * 50)
for categoria in resultado_inteiro.index:
    print(f"{categoria:<18}{resultado_pedacos[categoria]:>16,.2f}"
          f"{resultado_inteiro[categoria]:>16,.2f}")

diferenca = abs(resultado_pedacos.sum() - resultado_inteiro.sum())
print(f"\n   ✅ diferença: {diferenca:.6f}  (o mesmo resultado)")

print("""
⚠️ MAS NEM TODA AGREGAÇÃO SE DECOMPÕE ASSIM.

   ✅ soma, contagem, mínimo, máximo   → somam-se os parciais
   🔶 média                            → guarde soma E contagem
   🔴 mediana, percentil, distintos    → precisam de TODOS os dados

   💭 A mediana de pedaços não é a mediana do todo. Para essas, ou
      você carrega tudo, ou usa um algoritmo aproximado (t-digest), ou
      delega para um motor que faz isso — o que nos leva ao Parquet.
""")

## 4. 🎯 Parquet — o formato que resolve

In [ ]:
if not TEM_ARROW:
    print("⚠️ pyarrow indisponível")
else:
    PARQUET = BASE / "vendas.parquet"
    PARQUET_ZSTD = BASE / "vendas_zstd.parquet"

    otimizado.to_parquet(PARQUET, index=False)
    otimizado.to_parquet(PARQUET_ZSTD, index=False, compression="zstd")

    formatos = [
        ["CSV",              CSV.stat().st_size],
        ["Parquet (snappy)", PARQUET.stat().st_size],
        ["Parquet (zstd)",   PARQUET_ZSTD.stat().st_size],
    ]
    maior = max(t for _, t in formatos)
    print("O mesmo conjunto de dados:\n")
    for nome, bytes_ in formatos:
        barra = "█" * max(1, int(bytes_ / maior * 34))
        print(f"   {nome:<20}{tamanho(bytes_):>12}   {barra}")
    print(f"\n   Parquet zstd é {maior / PARQUET_ZSTD.stat().st_size:.1f}× menor que o CSV")

In [ ]:
if TEM_ARROW:
    print("E a leitura:\n")
    comparar([("CSV (com tipos)", lambda: pd.read_csv(CSV, dtype=TIPOS,
                                                      parse_dates=["data"])),
              ("Parquet completo", lambda: pd.read_parquet(PARQUET)),
              ("Parquet, 3 colunas",
               lambda: pd.read_parquet(PARQUET,
                                       columns=["categoria", "quantidade",
                                                "preco_unitario"]))],
             rotulo="leitura")

    print("""
🎯 A TERCEIRA LINHA É O PONTO DO FORMATO COLUNAR.

   `columns=[...]` não lê e descarta — ele NÃO LÊ as outras colunas
   do disco. Num arquivo de 40 colunas em que você usa 3, isso é
   quase toda a economia possível.

   ⚠️ E o CSV não consegue fazer isso. Para pegar a coluna 30, ele
      precisa percorrer as 29 anteriores de cada linha.
""")

In [ ]:
if TEM_ARROW:
    # 🔑 O Parquet guarda o SCHEMA — o CSV não
    import pyarrow.parquet as pq

    esquema = pq.read_schema(PARQUET)
    print("O Parquet guarda os tipos junto com os dados:\n")
    for campo in list(esquema)[:8]:
        print(f"   {campo.name:<18}{str(campo.type)}")

    print("""
🔑 E É POR ISSO QUE O PARQUET RESOLVE UM PROBLEMA QUE O CSV CRIA.

   Num CSV, `00123` vira o número 123 e o CEP perde o zero. `2026-01-05`
   vira texto ou data dependendo de quem lê. Toda leitura precisa
   ADIVINHAR — e adivinha diferente conforme a amostra.

   O Parquet não adivinha: o tipo está gravado.
""")

    # Provando
    (BASE / "cep.csv").write_text("cep,cidade\n01310-100,São Paulo\n00123,Teste\n",
                                  encoding="utf-8")
    lido = pd.read_csv(BASE / "cep.csv")
    print(f"   CSV: coluna 'cep' foi lida como {lido['cep'].dtype}")
    print(f"        valores: {lido['cep'].tolist()}")
    print("   🔴 se o CEP fosse só numérico, o zero à esquerda sumiria")

In [ ]:
if TEM_ARROW:
    # Particionamento — o que torna o "lago" navegável
    PARTICIONADO = BASE / "particionado"
    com_particao = otimizado.copy()
    com_particao["ano"] = com_particao["data"].dt.year
    com_particao["mes"] = com_particao["data"].dt.month
    com_particao.to_parquet(PARTICIONADO, partition_cols=["ano", "mes"], index=False)

    partes = sorted(PARTICIONADO.rglob("*.parquet"))
    print(f"{len(partes)} arquivos, organizados por pasta:\n")
    for p in partes[:4]:
        print(f"   {p.relative_to(PARTICIONADO)}")
    print("   ...")

    def tudo():
        return pd.read_parquet(PARTICIONADO)

    def so_um_mes():
        return pd.read_parquet(PARTICIONADO,
                               filters=[("ano", "==", 2026), ("mes", "==", 6)])

    print()
    comparar([("ler tudo", tudo), ("ler só junho/2026", so_um_mes)],
             rotulo="leitura particionada")

    print("""
🔑 PARTITION PRUNING: o motor lê os NOMES DAS PASTAS e pula os
   arquivos que não interessam. Ele nem abre os outros meses.

   💭 É o índice do M03 aplicado ao sistema de arquivos — e é por isso
      que a camada bronze da aula 10_01 é particionada por data de
      ingestão.
""")

## 5. 🎯 Polars — avaliação preguiçosa

In [ ]:
if not TEM_POLARS:
    print("⚠️ polars indisponível")
else:
    import polars as pl

    print("""
   PANDAS                          POLARS
   ══════                          ══════
   uma thread                      todos os núcleos
   índice                          sem índice
   avaliação IMEDIATA              🎯 PREGUIÇOSA (opcional)
   NumPy por baixo                 Arrow por baixo
   API acumulada em 15 anos        API desenhada de uma vez

   🎯 O diferencial não é "ser em Rust". É a AVALIAÇÃO PREGUIÇOSA:
      você descreve o que quer, e o motor OTIMIZA antes de executar.
""")

    df_pl = pl.read_parquet(PARQUET)
    print(f"   {df_pl.height:,} linhas × {df_pl.width} colunas")
    print(f"   memória: {tamanho(df_pl.estimated_size())}")

In [ ]:
if TEM_POLARS:
    # 🎯 O plano de consulta — o que o Polars faz por você
    consulta = (
        pl.scan_parquet(PARQUET)                       # 🔑 scan = preguiçoso
        .filter(pl.col("status") == "pago")
        .filter(pl.col("quantidade") >= 2)
        .with_columns(
            (pl.col("quantidade") * pl.col("preco_unitario")).alias("receita"))
        .group_by(["categoria", "canal"])
        .agg(pl.col("receita").sum().alias("receita_total"),
             pl.len().alias("vendas"))
        .sort("receita_total", descending=True)
    )

    print("O plano OTIMIZADO que o Polars vai executar:\n")
    print(consulta.explain())

> 🎯 **Leia o plano de baixo para cima — ele revela duas otimizações que você não escreveu.**
>
> **1. Projeção empurrada (*projection pushdown*).** A consulta usa 5 das 11 colunas. O plano mostra que só essas são lidas do arquivo. O Polars olhou o que você faria **depois** e decidiu o que ler **antes**.
>
> **2. Filtro empurrado (*predicate pushdown*).** Os dois `filter` foram combinados e empurrados para a leitura — as linhas descartadas nunca chegam a virar objeto na memória.
>
> 💭 **É exatamente o que um banco de dados faz há décadas** com `EXPLAIN QUERY PLAN` (M03). A novidade é ter isso num DataFrame.
>
> ⚠️ **E é por isso que `scan_parquet` é diferente de `read_parquet`:** o primeiro devolve um *plano*, o segundo devolve *dados*. Só o `.collect()` executa.

In [ ]:
if TEM_POLARS and TEM_ARROW:
    def com_pandas():
        d = pd.read_parquet(PARQUET)
        d = d[(d["status"] == "pago") & (d["quantidade"] >= 2)].copy()
        d["receita"] = d["quantidade"] * d["preco_unitario"]
        return (d.groupby(["categoria", "canal"], observed=True)
                .agg(receita_total=("receita", "sum"),
                     vendas=("receita", "size"))
                .sort_values("receita_total", ascending=False))

    def polars_imediato():
        d = pl.read_parquet(PARQUET)
        return (d.filter((pl.col("status") == "pago") & (pl.col("quantidade") >= 2))
                .with_columns((pl.col("quantidade") * pl.col("preco_unitario"))
                              .alias("receita"))
                .group_by(["categoria", "canal"])
                .agg(pl.col("receita").sum().alias("receita_total"),
                     pl.len().alias("vendas"))
                .sort("receita_total", descending=True))

    def polars_preguicoso():
        return consulta.collect()

    print("A MESMA pergunta, três caminhos:\n")
    comparar([("pandas", com_pandas),
              ("polars imediato", polars_imediato),
              ("polars preguiçoso", polars_preguicoso)],
             repeticoes=3, rotulo="motor")

> 🎯 **Repare na ordem: o Polars IMEDIATO pode sair mais lento que o pandas — e o preguiçoso ganha.**
>
> Isso confirma a tese da seção. Se a vantagem fosse "ser escrito em Rust", as duas versões do Polars ganhariam. **O que ganha é o plano de consulta:** ler 5 colunas de 11 e filtrar durante a leitura vale mais do que qualquer ganho de linguagem.
>
> ⚠️ E num conjunto deste tamanho os três respondem em dezenas de milissegundos. **A diferença só passa a importar quando o volume cresce** — ou quando essa consulta roda 500 vezes por dia.
>
> 💭 É o mesmo padrão do índice no M03: a estrutura que evita trabalho vence a força bruta bem executada.

In [ ]:
if TEM_POLARS:
    print("O resultado (idêntico nos três):\n")
    print(consulta.collect().head(6))

    print("""
💡 A SINTAXE DO POLARS EM TRÊS IDEIAS

   pl.col("x")            refere-se a uma coluna (é uma EXPRESSÃO)
   .filter(expr)          onde o pandas usa máscara booleana
   .with_columns(...)     onde o pandas usa atribuição
   .group_by().agg()      igual, mas com expressões

   🔑 A diferença conceitual: no Polars você escreve EXPRESSÕES que
      descrevem o cálculo. No pandas você manipula DADOS diretamente.
      É a expressão que permite ao motor otimizar.
""")

## 6. DuckDB — SQL sobre o arquivo

In [ ]:
if not TEM_DUCKDB:
    print("⚠️ duckdb indisponível")
else:
    import duckdb

    resultado = duckdb.sql(f"""
        SELECT categoria, canal,
               SUM(quantidade * preco_unitario) AS receita_total,
               COUNT(*) AS vendas
        FROM '{PARQUET}'
        WHERE status = 'pago' AND quantidade >= 2
        GROUP BY categoria, canal
        ORDER BY receita_total DESC
    """).df()

    print("SQL direto sobre o arquivo Parquet — sem carregar, sem importar:\n")
    print(resultado.head(6).to_string(index=False))

    print("""
🎯 REPARE: NENHUM `CREATE TABLE`, NENHUM `INSERT`.

   O DuckDB lê o Parquet como se fosse uma tabela. E ele faz as mesmas
   otimizações do Polars — lê só as colunas da consulta e empurra o
   filtro para a leitura.

   💡 E ele aceita glob:  FROM 'lago/bronze/*/*.parquet'
      Isso consulta o lago inteiro, com pruning de partição, numa linha.
""")

In [ ]:
if TEM_DUCKDB and TEM_ARROW:
    print("Consultando o LAGO PARTICIONADO com um glob:\n")
    r = duckdb.sql(f"""
        SELECT ano, mes, COUNT(*) AS vendas,
               ROUND(SUM(quantidade * preco_unitario), 2) AS receita
        FROM read_parquet('{PARTICIONADO}/**/*.parquet', hive_partitioning=true)
        WHERE status = 'pago' AND ano = 2026
        GROUP BY ano, mes ORDER BY mes
    """).df()
    print(r.to_string(index=False))

    print("""
🔑 `hive_partitioning=true` faz o DuckDB entender `ano=2026/mes=6/`
   como COLUNAS — e usá-las para pular arquivos.

   💭 É a mesma convenção que o Spark, o Athena e o BigQuery usam. Ela
      não é de nenhuma ferramenta: é do formato de pastas.
""")

## 7. O que escolher

In [ ]:
escolhas = [
    ["Explorar e limpar",        "pandas",         "ecossistema e familiaridade"],
    ["Pipeline em produção",     "polars",         "preguiçoso, previsível, rápido"],
    ["Agregação sobre arquivos", "duckdb",         "SQL, zero infraestrutura"],
    ["Não cabe na memória",      "polars/duckdb",  "os dois processam além da RAM"],
    ["Muito além de uma máquina", "Spark/nuvem",   "🔶 aí sim vale a complexidade"],
    ["Machine learning",         "pandas",         "é o que as bibliotecas esperam"],
]
tabela(["SITUAÇÃO", "FERRAMENTA", "POR QUÊ"], escolhas, [28, 16, 34])

print("""
💭 E A RESPOSTA HONESTA PARA A AURORA:

   300 mil vendas por ano cabem em ~10 MB de Parquet. Qualquer uma
   das três resolve, em milissegundos.

   🎯 A escolha aqui NÃO É por desempenho — é por CLAREZA e
      MANUTENÇÃO. Use o que a sua equipe lê melhor.

   ⚠️ E migrar tudo para Polars porque é mais rápido, quando o Pandas
      resolve em 50 ms, é otimizar o que não dói. O tempo gasto na
      migração seria melhor investido em qualidade de dado.
""")

In [ ]:
print("""
🔑 A ORDEM DE INVESTIGAÇÃO QUANDO ALGO ESTÁ LENTO OU ESTOURA A MEMÓRIA

   1. MEÇA. Qual etapa demora? Quanta memória usa?
      (sem medir, você otimiza o lugar errado)

   2. Declare os dtypes na leitura.
      Reduções de 50-90% por uma linha.

   3. Leia só as colunas que usa.
      `usecols=` no CSV, `columns=` no Parquet.

   4. Troque CSV por Parquet.
      Menor, mais rápido, e com o schema junto.

   5. Vetorize o que ainda for `apply`.
      (aula 10_02)

   6. Particione por data e filtre na leitura.

   7. SÓ ENTÃO troque de ferramenta.

   💭 Os passos 2 a 4 costumam resolver. Trocar de ferramenta é o
      último recurso, não o primeiro — e é o mais caro em tempo de
      equipe.
""")

## 🔧 Prática guiada — o relatório anual que estourava

In [ ]:
if TEM_ARROW and TEM_POLARS:
    print("A pergunta que dava MemoryError:\n")
    print("   'faturamento e margem por categoria, canal e mês, no ano'\n")

    def versao_original():
        """Como estava: lê o CSV inteiro, sem tipos."""
        d = pd.read_csv(CSV, parse_dates=["data"])
        d = d[d["status"] == "pago"].copy()
        d["receita"] = d["quantidade"] * d["preco_unitario"]
        d["margem"] = d["quantidade"] * (d["preco_unitario"] - d["custo_unitario"])
        d["mes"] = d["data"].dt.to_period("M").astype(str)
        return (d.groupby(["mes", "categoria", "canal"], observed=True)
                .agg(receita=("receita", "sum"), margem=("margem", "sum"))
                .reset_index())

    def versao_corrigida():
        """Parquet + Polars preguiçoso."""
        return (pl.scan_parquet(PARQUET)
                .filter(pl.col("status") == "pago")
                .with_columns([
                    (pl.col("quantidade") * pl.col("preco_unitario")).alias("receita"),
                    (pl.col("quantidade") * (pl.col("preco_unitario")
                                             - pl.col("custo_unitario"))).alias("margem"),
                    pl.col("data").dt.strftime("%Y-%m").alias("mes"),
                ])
                .group_by(["mes", "categoria", "canal"])
                .agg(pl.col("receita").sum(), pl.col("margem").sum())
                .sort("mes")
                .collect())

    medidos = comparar([("original (CSV + pandas)", versao_original),
                        ("corrigida (Parquet + Polars)", versao_corrigida)],
                       rotulo="versão")

    a, b = versao_original(), versao_corrigida()
    print(f"\n   linhas no resultado: pandas {len(a)} · polars {len(b)}")
    print(f"   receita total: R$ {a['receita'].sum():,.2f} vs "
          f"R$ {b['receita'].sum():,.2f}")
    print(f"   ✅ diferença: {abs(a['receita'].sum() - b['receita'].sum()):.2f}")

In [ ]:
print("Arquivos gerados nesta aula:\n")
for caminho in sorted(BASE.iterdir()):
    if caminho.is_file():
        print(f"   {caminho.name:<26}{tamanho(caminho.stat().st_size):>12}")
    else:
        total = sum(f.stat().st_size for f in caminho.rglob("*") if f.is_file())
        n = len(list(caminho.rglob("*.parquet")))
        print(f"   {caminho.name + '/':<26}{tamanho(total):>12}  ({n} arquivos)")

## 📝 Exercícios

**E1.** Meça a memória de cada coluna de um DataFrame seu. Identifique as três maiores.

**E2.** Calcule quanto uma string curta ocupa em Python e multiplique pelo número de linhas.

**E3.** 🎯 Escreva um `otimizar()` e meça a redução. Explique cada conversão.

**E4.** 🔴 Mostre que `float32` perde centavos numa soma de valores grandes. Explique por que dinheiro fica em `float64`.

**E5.** Compare `read_csv()` simples com `read_csv(dtype=...)` em tempo e memória.

**E6.** Implemente uma agregação em pedaços e prove que dá o mesmo resultado.

**E7.** 🔴 Explique por que mediana e contagem de distintos não se decompõem em pedaços.

**E8.** Salve o mesmo dado em CSV, Parquet snappy e Parquet zstd. Compare tamanho e tempo.

**E9.** Meça a diferença entre ler um Parquet inteiro e ler 3 colunas dele.

**E10.** 🔴 Mostre um CSV perdendo o zero à esquerda de um CEP. Mostre o Parquet preservando.

**E11.** Particione um Parquet por ano e mês. Meça o ganho ao filtrar um mês.

**E12.** 🎯 Escreva uma consulta Polars preguiçosa e leia o `explain()`. Aponte a projeção e o filtro empurrados.

**E13.** Compare pandas, Polars imediato e Polars preguiçoso na mesma pergunta.

**E14.** Escreva a mesma agregação em SQL com DuckDB, direto sobre o Parquet.

**E15.** Use `hive_partitioning=true` para consultar um lago particionado.

**E16.** Percorra a ordem de investigação nos seus dados e registre o ganho de cada passo.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

In [ ]:
# E16

## 📋 Cola de referência

```python
# ═══ Medir ═══
df.memory_usage(deep=True)        # 🔑 deep=True conta o texto de verdade
df.dtypes

# ═══ 🎯 dtypes ═══
pd.to_numeric(s, downcast="integer")   # int64 → int8/16/32
s.astype("category")                    # texto repetido → maior ganho
pd.read_csv(f, dtype={...}, usecols=[...])   # 🔑 evita o pico de memória
# 🔴 dinheiro fica em float64 — float32 perde centavo

# ═══ Pedaços ═══
for pedaco in pd.read_csv(f, chunksize=50_000):
    ...   # guarde só o AGREGADO
# ✅ soma/contagem/min/max   🔶 média (soma+contagem)
# 🔴 mediana/percentil/distintos NÃO se decompõem

# ═══ 🎯 Parquet ═══
df.to_parquet(f, compression="zstd")
pd.read_parquet(f, columns=[...])       # 🔑 não LÊ as outras
df.to_parquet(dir, partition_cols=["ano", "mes"])
pd.read_parquet(dir, filters=[("mes", "==", 6)])   # pruning
# guarda o SCHEMA → sem adivinhação, sem perder zero à esquerda

# ═══ 🎯 Polars preguiçoso ═══
(pl.scan_parquet(f)                     # scan = plano, não dados
 .filter(pl.col("x") > 0)
 .with_columns((pl.col("a") * pl.col("b")).alias("c"))
 .group_by("k").agg(pl.col("c").sum())
 .collect())                            # 🔑 só aqui executa
consulta.explain()                      # veja projeção e filtro empurrados

# ═══ DuckDB ═══
duckdb.sql("SELECT ... FROM 'arquivo.parquet' WHERE ...").df()
duckdb.sql("... FROM read_parquet('lago/**/*.parquet', hive_partitioning=true)")

# ═══ Ordem de investigação ═══
# 1 medir · 2 dtypes · 3 só as colunas · 4 Parquet
# 5 vetorizar · 6 particionar · 7 SÓ ENTÃO trocar de ferramenta
```

## ✅ Checklist de saída

**Memória**

- [ ] Meço a memória com `deep=True`
- [ ] Sei por que texto como `object` é caro
- [ ] Uso `category` para texto repetido
- [ ] Faço downcast de inteiros
- [ ] 🔴 **Mantenho dinheiro em `float64`**
- [ ] Declaro `dtype` na leitura, não depois

**Volume**

- [ ] Sei processar em pedaços
- [ ] 🔴 **Sei quais agregações não se decompõem**

**Formato**

- [ ] Uso Parquet em vez de CSV para dado intermediário
- [ ] Leio só as colunas que preciso
- [ ] Sei que o Parquet guarda o schema
- [ ] Particiono por data e filtro na leitura

**Motores**

- [ ] 🎯 **Sei ler um plano de consulta do Polars**
- [ ] Sei o que são projeção e filtro empurrados
- [ ] Sei a diferença entre `scan_` e `read_`
- [ ] Sei consultar Parquet com SQL no DuckDB

**Julgamento**

- [ ] 💭 **Sigo a ordem de investigação antes de trocar de ferramenta**
- [ ] Sei que para a Aurora as três resolvem — e escolho por clareza

---

### ➡️ Próxima aula

**`10_04_Extracao_e_Web_Scraping.ipynb`** — De onde os dados vêm: arquivos, APIs, bancos com ingestão incremental, e páginas web (com as regras que ninguém lê).